<a href="https://colab.research.google.com/github/Suim00/hdac-knhanes-skipbreakfast-obesity/blob/main/code/09_data_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 9주차 자료 확인 (9팀)


## 1. 내 Google Drive 연결 및 자료 읽기

In [42]:
# 1) 내 Google Drive 연결 - 원시자료는 각자 자기 Drive에 둡니다
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/hdac_data/HN24_ALL(SAS)'   # 팀 공통 경로
DATA_FILE = f'{DATA_DIR}/hn24_all.sas7bdat'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [43]:
# 2) 필요한 변수만 읽기
!pip -q install pyreadstat
import pyreadstat
import pandas as pd

# 변수 정의표의 변수명 = 이용지침서 표기 (노출변수는 L_BR_FQ, 이용지침서 218쪽. BH9_11은 독감 예방접종 변수)
need_core = ['age', 'sex', 'HE_prg', 'L_BR_FQ', 'HE_ht', 'HE_wt', 'HE_BMI', 'HE_obe', 'wt_tot', 'kstrata', 'psu']
need_cov  = ['town_t', 'educ', 'sm_presnt', 'dr_month', 'N_DIET', 'N_EN']   # 변수 정의표의 공변량
need = need_core + need_cov

# 파일의 전체 변수 목록만 먼저 읽기 (자료 행은 읽지 않음)
_, meta0 = pyreadstat.read_sas7bdat(DATA_FILE, metadataonly=True)
all_cols = list(meta0.column_names)
print('파일 전체 변수 수:', len(all_cols))

파일 전체 변수 수: 813


## 2. 변수 존재 여부

In [44]:
lower_map = {c.lower(): c for c in all_cols}
present = {v: lower_map.get(v.lower()) for v in need}
for v, actual in present.items():
    tag = '[핵심]' if v in need_core else '[공변량]'
    print(f'{tag} {v:10s} ->', actual if actual else '파일에 없음')

missing_vars = [v for v, a in present.items() if a is None]
print()
print('없는 변수:', missing_vars if missing_vars else '없음')

# 없는 변수는 비슷한 이름의 후보를 찾아 정의표 수정에 참고한다
for v in missing_vars:
    key = v.lower().split('_')[0][:4]
    cands = [c for c in all_cols if key in c.lower()]
    print(f'  {v} 유사 후보:', cands[:15] if cands else '없음')

[핵심] age        -> age
[핵심] sex        -> sex
[핵심] HE_prg     -> HE_prg
[핵심] L_BR_FQ    -> L_BR_FQ
[핵심] HE_ht      -> HE_ht
[핵심] HE_wt      -> HE_wt
[핵심] HE_BMI     -> HE_BMI
[핵심] HE_obe     -> HE_obe
[핵심] wt_tot     -> wt_tot
[핵심] kstrata    -> kstrata
[핵심] psu        -> psu
[공변량] town_t     -> town_t
[공변량] educ       -> educ
[공변량] sm_presnt  -> sm_presnt
[공변량] dr_month   -> dr_month
[공변량] N_DIET     -> N_DIET
[공변량] N_EN       -> N_EN

없는 변수: 없음


In [45]:
# 파일에 있는 이름으로 변수만 골라 읽기
use_cols = [a for a in present.values() if a]
df, meta = pyreadstat.read_sas7bdat(DATA_FILE, usecols=use_cols)
# 이후 코드에서 쓰기 쉽도록 정의표의 이름으로 통일
df = df.rename(columns={a: v for v, a in present.items() if a})
print(df.shape)   # 행 수와 열 수만 출력

(6997, 17)


## 3. 원 코드별 빈도표

In [46]:
# 노출: 아침식사 빈도 (L_BR_FQ, 이용지침서 218쪽)
# 1 주5~7회, 2 주3~4회, 3 주1~2회, 4 거의 안한다(주0회), 9 모름/무응답
# 이분형: 결식(노출군) = 3, 4 / 아침식사(비교군) = 1, 2
EXPOSED_CODES   = [3, 4]
REFERENCE_CODES = [1, 2]
assert not set(EXPOSED_CODES) & set(REFERENCE_CODES)

print('L_BR_FQ 원 코드별 인원 (결측 포함)')
print(df['L_BR_FQ'].value_counts(dropna=False).sort_index())
print()

df['breakfast_skip'] = df['L_BR_FQ'].map(
    lambda x: 1 if x in EXPOSED_CODES else (0 if x in REFERENCE_CODES else None))

print('이분형 breakfast_skip 인원 (전체 참여자, 결측 포함)')
print(df['breakfast_skip'].value_counts(dropna=False).sort_index()
        .rename({0.0: '0 아침식사(주 3회 이상)', 1.0: '1 결식(주 0~2회)'}))

L_BR_FQ 원 코드별 인원 (결측 포함)
L_BR_FQ
1.0    4214
2.0     669
3.0     786
4.0    1136
NaN     192
Name: count, dtype: int64

이분형 breakfast_skip 인원 (전체 참여자, 결측 포함)
breakfast_skip
0 아침식사(주 3회 이상)    4883
1 결식(주 0~2회)       1922
NaN                 192
Name: count, dtype: int64


In [47]:
# L_BR_FQ 결측(NaN)이 어떤 사람들인지 연령대별로 확인 (집계값). 무응답(9)은 이번 자료에 관측되지 않음
age_band = pd.cut(df['age'], [-1, 0, 11, 18, 29, 39, 49, 59, 69, 120],
                  labels=['0세', '1~11세', '12~18세', '19~29세', '30~39세', '40~49세', '50~59세', '60~69세', '70세+'])
print(pd.crosstab(age_band, df['L_BR_FQ'].isna().map({True: '결측', False: '응답'})))
print('L_BR_FQ = 9 (모름/무응답) 인원:', int((df['L_BR_FQ'] == 9).sum()))

L_BR_FQ  결측    응답
age              
1~11세    10   556
12~18세    8   390
19~29세   15   695
30~39세   14   629
40~49세   29   906
50~59세   43  1085
60~69세   38  1242
70세+     35  1302
L_BR_FQ = 9 (모름/무응답) 인원: 0


In [48]:
# 결과: 비만 유병여부 (HE_obe) - 1 저체중, 2 정상, 3 비만전단계, 4 1단계, 5 2단계, 6 3단계 (비임신자만 값 있음)
print('HE_obe 원 코드별 인원 (결측 포함)')
print(df['HE_obe'].value_counts(dropna=False).sort_index())

HE_obe 원 코드별 인원 (결측 포함)
HE_obe
1.0     285
2.0    2134
3.0    1329
4.0    1751
5.0     344
6.0      66
NaN    1088
Name: count, dtype: int64


In [49]:
# 성별 (sex) - 1 남자, 2 여자
print('sex 원 코드별 인원')
print(df['sex'].value_counts(dropna=False).sort_index())

sex 원 코드별 인원
sex
1.0    3070
2.0    3927
Name: count, dtype: int64


In [50]:
# 임신여부 (HE_prg) - 0 아니오, 1 예, 8 비해당(남자)
print('HE_prg 원 코드별 인원 (결측 포함)')
print(df['HE_prg'].value_counts(dropna=False).sort_index())

HE_prg 원 코드별 인원 (결측 포함)
HE_prg
0.0    3911
1.0      16
8.0    3070
Name: count, dtype: int64


## 4. 가중치·층·집락 정보

In [51]:
print('wt_tot 결측 인원:', int(df['wt_tot'].isna().sum()))
print('wt_tot 0 이하 인원:', int((df['wt_tot'] <= 0).sum()))
print()
print('wt_tot 요약 (집계값)')
print(df['wt_tot'].describe())
print()
print('kstrata 층 수:', df['kstrata'].nunique(dropna=True), '/ 결측:', int(df['kstrata'].isna().sum()))
print('psu 집락 수:', df['psu'].nunique(dropna=True), '/ 결측:', int(df['psu'].isna().sum()))

wt_tot 결측 인원: 192
wt_tot 0 이하 인원: 0

wt_tot 요약 (집계값)
count     6805.000000
mean      7571.202792
std       3537.288651
min        388.119306
25%       5138.748407
50%       6786.485519
75%       9405.572181
max      19285.643829
Name: wt_tot, dtype: float64

kstrata 층 수: 27 / 결측: 0
psu 집락 수: 192 / 결측: 0


## 5. 분석 대상 선정 단계별 인원

- 0단계: 전체 조사 참여자
- 1단계: 만 19세 이상 성인
- 2단계: 임신 중인 여성 제외 (`HE_prg` = 1)
- 3단계: 아침식사 빈도 응답자 (`L_BR_FQ` 1~4, 모름·무응답 제외)
- 4단계: 신장·체중·BMI 결측 제외 (n=94). 이상값 기준은 적용하지 않았다 (`BMI_MIN` = `BMI_MAX` = None). 94명은 모두 신장(`HE_ht`)과 BMI 결측이며, 체중 결측 23명은 이 94명에 포함된다.
- 5단계: 가중치·층·집락 변수가 유효한 자료

In [52]:
# 4단계 이상값 확인용 - 신장·체중·BMI 분포 요약 (분석 대상 중 3단계까지 통과한 사람, 집계값)
chk = (df['age'] >= 19) & (df['HE_prg'] != 1) & df['L_BR_FQ'].isin([1, 2, 3, 4])
print(df.loc[chk, ['HE_ht', 'HE_wt', 'HE_BMI']].describe().round(1))

        HE_ht   HE_wt  HE_BMI
count  5749.0  5820.0  5749.0
mean    163.4    64.8    24.1
std       9.4    13.6     3.9
min     127.9    27.5    13.1
25%     156.7    54.9    21.5
50%     163.0    63.0    23.7
75%     170.0    72.7    26.3
max     194.0   147.0    48.7


In [53]:
# 4단계 제외(94명)의 내용 확인: 3단계까지 통과한 사람 중 변수별 결측 인원
base = (df['age'] >= 19) & (df['HE_prg'] != 1) & df['L_BR_FQ'].isin([1, 2, 3, 4])
print('3단계까지 통과:', int(base.sum()))
print(df.loc[base, ['HE_ht', 'HE_wt', 'HE_BMI']].isna().sum().to_string())
print('신장·체중·BMI 중 하나라도 결측:', int(df.loc[base, ['HE_ht', 'HE_wt', 'HE_BMI']].isna().any(axis=1).sum()))

3단계까지 통과: 5843
HE_ht     94
HE_wt     23
HE_BMI    94
신장·체중·BMI 중 하나라도 결측: 94


In [54]:
# 4단계 이상값 기준: 이용지침서에는 신장·체중 이상값 제외 기준이 없다.
# 현재는 BMI_MIN = BMI_MAX = None 이므로 이상값으로 제외된 사람은 없고,
# 4단계 제외 인원은 신장·체중·BMI 결측만으로 발생한다.
# 팀에서 기준을 정하면 아래 값을 채운다. None이면 이상값 기준으로는 제외하지 않는다.
BMI_MIN = None   # 예: 팀이 정한 BMI 하한
BMI_MAX = None   # 예: 팀이 정한 BMI 상한

In [55]:
steps = []
cur = pd.Series(True, index=df.index)

def add_step(name, cond):
    global cur
    before = int(cur.sum())
    cur = cur & cond
    after = int(cur.sum())
    steps.append({'단계': name, '제외된 인원': before - after, '남은 인원': after})

steps.append({'단계': '0. 전체 조사 참여자', '제외된 인원': 0, '남은 인원': int(cur.sum())})
add_step('1. 만 19세 이상 성인', df['age'] >= 19)
add_step('2. 임신 중인 여성 제외 (HE_prg = 1)', df['HE_prg'] != 1)
add_step('3. 아침식사 빈도 응답자 (L_BR_FQ 1~4)', df['L_BR_FQ'].isin([1, 2, 3, 4]))

ok_body = df['HE_ht'].notna() & df['HE_wt'].notna() & df['HE_BMI'].notna()
if BMI_MIN is not None:
    ok_body = ok_body & (df['HE_BMI'] >= BMI_MIN)
if BMI_MAX is not None:
    ok_body = ok_body & (df['HE_BMI'] <= BMI_MAX)
add_step('4. 신장·체중·BMI 결측 제외' + ('' if (BMI_MIN is None and BMI_MAX is None) else ' + BMI 이상값 제외'), ok_body)

ok_design = (df['wt_tot'].notna() & (df['wt_tot'] > 0)
             & df['kstrata'].notna() & df['psu'].notna())
add_step('5. 가중치·층·집락 변수 유효', ok_design)

flow = pd.DataFrame(steps)
print(flow.to_string(index=False))
print()
print('최종 분석대상:', int(cur.sum()))
print('최종 분석대상 중 HE_obe 결측:', int(df.loc[cur, 'HE_obe'].isna().sum()))

                          단계  제외된 인원  남은 인원
                0. 전체 조사 참여자       0   6997
              1. 만 19세 이상 성인     964   6033
 2. 임신 중인 여성 제외 (HE_prg = 1)      16   6017
3. 아침식사 빈도 응답자 (L_BR_FQ 1~4)     174   5843
          4. 신장·체중·BMI 결측 제외      94   5749
           5. 가중치·층·집락 변수 유효       0   5749

최종 분석대상: 5749
최종 분석대상 중 HE_obe 결측: 0


## 5-1. 아침 결식 이분형 인원 (분석 대상 안)
5번 셀에서 만든 최종 분석대상(`cur`) 안에서 아침 결식(노출군)과 아침식사(비교군)의 인원을 확인한다. 표본 인원 기준이며 가중치를 적용하지 않은 값이다.

| `L_BR_FQ` 원 코드 | 의미 | 이분형 |
|---|---|---|
| 1 | 주 5~7회 | 0 아침식사(비교군) |
| 2 | 주 3~4회 | 0 아침식사(비교군) |
| 3 | 주 1~2회 | 1 결식(노출군) |
| 4 | 거의 안한다(주0회) | 1 결식(노출군) |
| 9, 결측 | 모름/무응답, 비해당 | 분석 제외 |

In [56]:
sub = df[cur].copy()
print('분석 대상 인원:', len(sub))
print()

cnt = sub['breakfast_skip'].value_counts(dropna=False).sort_index()
pct = (cnt / cnt.sum() * 100).round(1)
out = pd.DataFrame({'인원': cnt.astype(int), '비율(%)': pct})
out.index = ['0 아침식사(주 3회 이상)', '1 결식(주 0~2회)']
print('breakfast_skip 인원 (분석 대상)')
print(out)

분석 대상 인원: 5749

breakfast_skip 인원 (분석 대상)
                   인원  비율(%)
0 아침식사(주 3회 이상)  4063   70.7
1 결식(주 0~2회)     1686   29.3


In [57]:
# 이분형 기준 비만(HE_obe 4~6, BMI 25 이상) 인원 - 가중치를 적용하지 않은 단순 집계 (설명용)
sub['obese'] = (sub['HE_obe'] >= 4).astype(int)
tab = pd.crosstab(sub['breakfast_skip'], sub['obese'])
tab.columns = ['비비만(BMI<25)', '비만(BMI>=25)']
tab.index = ['0 아침식사(주 3회 이상)', '1 결식(주 0~2회)']
tab['합계'] = tab.sum(axis=1)
tab['비만 비율(%)'] = (tab['비만(BMI>=25)'] / tab['합계'] * 100).round(1)
print(tab)

                 비비만(BMI<25)  비만(BMI>=25)    합계  비만 비율(%)
0 아침식사(주 3회 이상)         2600         1463  4063      36.0
1 결식(주 0~2회)            1050          636  1686      37.7


### 비가중 비교 결과 해석 시 주의
- 위 표는 가중치를 적용하지 않은 단순 집계(기술통계)이다.
- 비만 비율은 아침 식사군 36.0%, 결식군 37.7%이다.
- 4범주로 나누면 주 5~7회 35.1%, 주 3~4회 41.7%, 주 1~2회 40.0%, 거의 안 함 36.2%로, 결식 빈도가 늘수록 비만 비율이 일관되게 증가하는 형태는 아니다.
- 연구계획서의 예상 방향(결식 → 비만 증가)은 사전 가설로 유지하며, 실제 분석 결과와 구분한다.
- 관련성은 가중치·복합표본설계와 교란변수 보정을 적용한 분석 결과로 해석한다.

## 6. 공변량 확인
변수 정의표의 공변량(`town_t`, `educ`, `sm_presnt`, `dr_month`, `N_DIET`, `N_EN`)의 원 코드와 결측을 확인한다.

In [58]:
# 공변량 원 코드별 인원 (분석 대상 안, 결측 포함). 범주가 많으면 요약통계로 대체
cov_in_df = [v for v in need_cov if v in sub.columns]
for v in cov_in_df:
    print(f'--- {v} ---')
    if sub[v].nunique(dropna=True) <= 15:
        print(sub[v].value_counts(dropna=False).sort_index())
    else:
        print(sub[v].describe().round(2))
        print('결측:', int(sub[v].isna().sum()))
    print()
not_found = [v for v in need_cov if v not in cov_in_df]
print('파일에 없어 확인하지 못한 공변량:', not_found if not_found else '없음')

--- town_t ---
town_t
1.0    4595
2.0    1154
Name: count, dtype: int64

--- educ ---
educ
1.0        4
2.0      109
3.0      666
4.0      603
5.0     1517
6.0      835
7.0     1539
8.0      328
99.0     148
Name: count, dtype: int64

--- sm_presnt ---
sm_presnt
0.0    4847
1.0     761
NaN     141
Name: count, dtype: int64

--- dr_month ---
dr_month
0.0    2729
1.0    2880
NaN     140
Name: count, dtype: int64

--- N_DIET ---
N_DIET
1.0    1862
2.0    3885
NaN       2
Name: count, dtype: int64

--- N_EN ---
count    5747.00
mean     1796.16
std       791.91
min       161.18
25%      1246.52
50%      1669.06
75%      2190.63
max      7187.77
Name: N_EN, dtype: float64
결측: 2

파일에 없어 확인하지 못한 공변량: 없음


In [59]:
# 공변량 무응답·비해당 코드를 결측으로 처리 (코드 기준은 이용지침서 134·232쪽)
# educ: 88 비해당(소아), 99 모름·무응답 -> 결측
# N_DIET: 9 모름/무응답 -> 결측
# town_t, sm_presnt, dr_month: 별도의 무응답 코드 없음 (원값 그대로 사용)
sub['educ_c']   = sub['educ'].where(~sub['educ'].isin([88, 99]))
sub['N_DIET_c'] = sub['N_DIET'].where(~sub['N_DIET'].isin([9]))

cov_final = ['town_t', 'educ_c', 'sm_presnt', 'dr_month', 'N_DIET_c', 'N_EN']
cov_final = [v for v in cov_final if v in sub.columns]

print('최종 분석 대상:', len(sub))
print()
print('공변량별 결측 인원 (처리 후)')
print(sub[cov_final].isna().sum().to_string())

complete = sub[cov_final].notna().all(axis=1)
print()
print('공변량 모두 값이 있는 인원(완전자료):', int(complete.sum()))
print('공변량 결측으로 빠질 수 있는 인원:', int((~complete).sum()))

최종 분석 대상: 5749

공변량별 결측 인원 (처리 후)
town_t         0
educ_c       148
sm_presnt    141
dr_month     140
N_DIET_c       2
N_EN           2

공변량 모두 값이 있는 인원(완전자료): 5600
공변량 결측으로 빠질 수 있는 인원: 149


### 분석 대상과 복합표본설계 적용 메모
- 최종 분석 대상 5,749명의 선정 기준은 유지한다.
- 공변량 결측 처리(`educ` 99, `N_DIET` 9를 결측으로 처리) 후 공변량이 모두 있는 대상자는 5,600명이며, 149명(`educ` 148명, `sm_presnt` 141명, `dr_month` 140명, `N_DIET` 2명, `N_EN` 2명, 중복 포함)은 회귀분석에서 제외될 수 있다.
- 후속 분석에서는 `wt_tot`(가중치), `kstrata`(층), `psu`(집락)를 실제 분석 방법에 반영한다.
- 특정 연령대나 성별 결과를 따로 제시할 때는 자료를 잘라 따로 분석하지 말고, 복합표본설계의 도메인(부분모집단) 분석을 고려한다.